# 02C — ABCD Matrix Treatment of a Fabry–Pérot Cavity

## Purpose

In 02B we described the Gaussian mode of a two-mirror cavity using direct formulas. That approach is especially simple for a symmetric cavity.

Here we introduce the **ABCD matrix method**, which provides a systematic way to propagate rays and, more importantly, the complex Gaussian beam parameter \(q\).

This notebook treats:

1. free-space propagation,
2. reflection from spherical mirrors,
3. the round-trip ABCD matrix,
4. the self-consistent Gaussian eigenmode,
5. the symmetric cavity as a validation case,
6. an asymmetric cavity where the waist moves away from the centre.

This is the method that will later be generalized to the four-mirror bow-tie cavity.

## 1. The ABCD ray representation

In paraxial optics, a ray can be represented by

\[
\mathbf r=
\begin{pmatrix}
x\\
\theta
\end{pmatrix},
\]

where \(x\) is the transverse position and \(\theta\) is the paraxial angle.

An optical element is represented by a matrix

\[
M=
\begin{pmatrix}
A&B\\
C&D
\end{pmatrix},
\]

and the ray transforms according to

\[
\mathbf r_{\rm out}=M\mathbf r_{\rm in}.
\]

The same ABCD matrices also act on the complex Gaussian beam parameter through

\[
\boxed{
q_{\rm out}
=
\frac{Aq_{\rm in}+B}
{Cq_{\rm in}+D}
}.
\]

In [1]:
import numpy as np
import matplotlib.pyplot as plt

c = 299_792_458.0
lambda0 = 1064e-9

L = 0.50
ROC1 = 1.00
ROC2 = 1.00

print(f"Wavelength = {lambda0*1e9:.1f} nm")
print(f"Cavity length = {L:.3f} m")
print(f"ROC1 = {ROC1:.3f} m")
print(f"ROC2 = {ROC2:.3f} m")

/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-2htsd924 because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Wavelength = 1064.0 nm
Cavity length = 0.500 m
ROC1 = 1.000 m
ROC2 = 1.000 m


## 2. Free-space propagation matrix

For propagation through a distance \(d\),

\[
\boxed{
P(d)=
\begin{pmatrix}
1&d\\
0&1
\end{pmatrix}
}.
\]

The matrix gives

\[
\begin{pmatrix}
x'\\
\theta'
\end{pmatrix}
=
P(d)
\begin{pmatrix}
x\\
\theta
\end{pmatrix}.
\]

For example, a ray with \(x=1\) mm and \(\theta=2\) mrad changes its transverse position during propagation while its paraxial angle remains unchanged.

In [2]:
P_L = np.array([
    [1.0, L],
    [0.0, 1.0]
])

print("Free-space propagation matrix:")
print(P_L)

x0 = 1e-3
theta0 = 2e-3

ray0 = np.array([
    [x0],
    [theta0]
])

ray1 = P_L @ ray0

print("\nInitial ray [x, theta]^T:")
print(ray0)

print("\nRay after propagation:")
print(ray1)

Free-space propagation matrix:
[[1.  0.5]
 [0.  1. ]]

Initial ray [x, theta]^T:
[[0.001]
 [0.002]]

Ray after propagation:
[[0.002]
 [0.002]]


## 3. Spherical mirror matrix

For a spherical mirror of radius of curvature \(R\), the paraxial reflection matrix is

\[
\boxed{
M(R)=
\begin{pmatrix}
1&0\\
-\frac{2}{R}&1
\end{pmatrix}
}.
\]

The factor of 2 appears because reflection from a spherical mirror changes the propagation direction by twice the surface slope.

For the symmetric cavity,

\[
M_1=M_2=M(R).
\]

In [3]:
M_1 = np.array([
    [1.0, 0.0],
    [-2.0/ROC1, 1.0]
])

M_2 = np.array([
    [1.0, 0.0],
    [-2.0/ROC2, 1.0]
])

print("Mirror 1 matrix:")
print(M_1)

print("\nMirror 2 matrix:")
print(M_2)

Mirror 1 matrix:
[[ 1.  0.]
 [-2.  1.]]

Mirror 2 matrix:
[[ 1.  0.]
 [-2.  1.]]


## 4. Round-trip matrix

Choose the reference plane **immediately after reflection from mirror 1**.

The sequence for one complete round trip is:

1. propagate from M1 to M2,
2. reflect from M2,
3. propagate from M2 back to M1,
4. reflect from M1.

Therefore

\[
\boxed{
M_{\rm RT}
=
M_1P(L)M_2P(L)
}.
\]

The order matters because matrix multiplication is not commutative.

In [4]:
M_rt = M_1 @ P_L @ M_2 @ P_L

print("Round-trip ABCD matrix:")
print(M_rt)

A, B, C, D = M_rt.flatten()

print(f"\nA = {A:.6f}")
print(f"B = {B:.6f} m")
print(f"C = {C:.6f} 1/m")
print(f"D = {D:.6f}")

Round-trip ABCD matrix:
[[ 0.   0.5]
 [-2.  -1. ]]

A = 0.000000
B = 0.500000 m
C = -2.000000 1/m
D = -1.000000


## 5. Self-consistent Gaussian eigenmode

A cavity eigenmode must reproduce itself after one round trip:

\[
q_{\rm out}=q_{\rm in}=q.
\]

Using the ABCD transformation,

\[
q=\frac{Aq+B}{Cq+D}.
\]

Multiplying through gives

\[
Cq^2+(D-A)q-B=0.
\]

Thus the Gaussian eigenmode is obtained from the quadratic equation

\[
\boxed{
Cq^2+(D-A)q-B=0
}.
\]

There are two mathematical roots. For a physical Gaussian beam, the convention

\[
q=z+iz_R
\]

requires

\[
\boxed{\operatorname{Im}(q)>0}.
\]

The other root is the mathematical conjugate and does not represent the physical forward Gaussian mode under this convention.

In [5]:
discriminant = (D - A)**2 + 4*B*C

q_plus = (
    -(D - A) + np.sqrt(discriminant + 0j)
) / (2*C)

q_minus = (
    -(D - A) - np.sqrt(discriminant + 0j)
) / (2*C)

print(f"First root  q+ = {q_plus:.9f} m")
print(f"Second root q- = {q_minus:.9f} m")

q_eigen = q_plus if q_plus.imag > 0 else q_minus

print(f"\nPhysical eigenmode q = {q_eigen:.9f} m")
print(f"Im(q) > 0? {q_eigen.imag > 0}")

First root  q+ = -0.250000000-0.433012702j m
Second root q- = -0.250000000+0.433012702j m

Physical eigenmode q = -0.250000000+0.433012702j m
Im(q) > 0? True


## 6. Connection with the Gaussian-beam definition of \(q\)

The complex beam parameter is conventionally written as

\[
\boxed{
\frac{1}{q(z)}
=
\frac{1}{R(z)}
-i\frac{\lambda_0}{\pi w^2(z)}
}.
\]

Equivalently,

\[
q(z)=z+iz_R.
\]

Therefore

\[
\operatorname{Re}(q)=z
\]

measures the longitudinal distance from the waist, while

\[
\operatorname{Im}(q)=z_R
\]

is the Rayleigh range when the coordinate origin is at the waist.

The waist radius follows from

\[
\boxed{
w_0=
\sqrt{\frac{\lambda_0z_R}{\pi}}
}.
\]

In [6]:
z_R_abcd = q_eigen.imag
z_waist_from_M1 = -q_eigen.real

w0_abcd = np.sqrt(lambda0 * z_R_abcd / np.pi)

z_R_direct = np.sqrt(
    (L/2) * (ROC1 - L/2)
)

w0_direct = np.sqrt(
    lambda0 * z_R_direct / np.pi
)

print(f"ABCD Rayleigh range = {z_R_abcd:.9f} m")
print(f"Direct Rayleigh range = {z_R_direct:.9f} m")
print(f"Rayleigh-range agreement = {np.isclose(z_R_abcd, z_R_direct)}")

print(f"\nWaist position from M1 = {z_waist_from_M1:.9f} m")
print(f"Expected symmetric waist position = {L/2:.9f} m")

print(f"\nABCD waist radius = {w0_abcd*1e3:.9f} mm")
print(f"Direct waist radius = {w0_direct*1e3:.9f} mm")

ABCD Rayleigh range = 0.433012702 m
Direct Rayleigh range = 0.433012702 m
Rayleigh-range agreement = True

Waist position from M1 = 0.250000000 m
Expected symmetric waist position = 0.250000000 m

ABCD waist radius = 0.382953635 mm
Direct waist radius = 0.382953635 mm


## 7. Beam propagation from the ABCD eigenmode

Once the self-consistent \(q\) parameter is known at the reference plane, we can propagate it through the cavity.

For free-space propagation over a distance \(d\),

\[
q(d)=q_0+d.
\]

The beam radius can be recovered from

\[
\frac{1}{q}
=
\frac{1}{R}
-i\frac{\lambda_0}{\pi w^2}.
\]

Therefore,

\[
\boxed{
w=
\sqrt{
-\frac{\lambda_0}
{\pi\,\operatorname{Im}(1/q)}
}
}.
\]

In [7]:
def propagate_q(q, matrix):
    A, B, C, D = matrix.flatten()
    return (A*q + B) / (C*q + D)

def beam_radius(q, wavelength):
    return np.sqrt(
        -wavelength /
        (np.pi * (1/q).imag)
    )

z = np.linspace(0, L, 1000)

q_z = np.array([
    propagate_q(q_eigen, np.array([[1.0, distance],
                                   [0.0, 1.0]]))
    for distance in z
])

w_z = np.array([
    beam_radius(q, lambda0)
    for q in q_z
])

plt.figure(figsize=(9, 4))
plt.plot(z, w_z*1e3)
plt.xlabel("Position from M1 [m]")
plt.ylabel("Beam radius w(z) [mm]")
plt.title("Gaussian eigenmode from ABCD propagation")
plt.grid(True)
plt.show()

/tmp/ipykernel_1055/2104604600.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Round-trip closure check

The defining property of the eigenmode is

\[
q_{\rm after\ round\ trip}=q_{\rm before\ round\ trip}.
\]

We can verify this numerically by applying the complete round-trip matrix to the eigenmode:

\[
q_{\rm RT}
=
\frac{Aq+B}{Cq+D}.
\]

Because this is a numerical calculation, we use `np.isclose` rather than exact `==` comparison.

In [8]:
q_after_rt = propagate_q(q_eigen, M_rt)

print(f"Initial q = {q_eigen:.12f} m")
print(f"Round-trip q = {q_after_rt:.12f} m")
print(f"Round-trip closure? {np.isclose(q_after_rt, q_eigen)}")
print(f"Difference = {q_after_rt - q_eigen:.3e} m")

Initial q = -0.250000000000+0.433012701892j m
Round-trip q = -0.250000000000+0.433012701892j m
Round-trip closure? True
Difference = -5.551e-17+5.551e-17j m


## 9. Asymmetric cavity: \(ROC_1\neq ROC_2\)

Now change one mirror curvature while keeping the cavity length fixed:

\[
ROC_1=1.00\ {\rm m},
\qquad
ROC_2=0.75\ {\rm m}.
\]

The direct symmetric formulas from 02B are no longer sufficient.

The ABCD method, however, does not require the cavity to be symmetric. We simply construct the two different mirror matrices and form the new round-trip matrix:

\[
M_{\rm RT}=M_1P(L)M_2P(L).
\]

The eigenmode is again obtained from

\[
Cq^2+(D-A)q-B=0.
\]

In [9]:
ROC1 = 1.00
ROC2 = 0.75

M_1_asym = np.array([
    [1.0, 0.0],
    [-2.0/ROC1, 1.0]
])

M_2_asym = np.array([
    [1.0, 0.0],
    [-2.0/ROC2, 1.0]
])

M_rt_asym = M_1_asym @ P_L @ M_2_asym @ P_L

A_asym, B_asym, C_asym, D_asym = M_rt_asym.flatten()

discriminant_asym = (
    (D_asym - A_asym)**2
    + 4*B_asym*C_asym
)

roots_asym = np.roots([
    C_asym,
    D_asym - A_asym,
    -B_asym
])

q_asym = next(q for q in roots_asym if q.imag > 0)

z_asym = q_asym.real
zR_asym = q_asym.imag
w0_asym = np.sqrt(lambda0 * zR_asym / np.pi)

print("Asymmetric cavity:")
print(f"ROC1 = {ROC1:.3f} m")
print(f"ROC2 = {ROC2:.3f} m")
print(f"\nq eigenmode = {q_asym:.9f} m")
print(f"Waist position measured from M1 = {-z_asym:.9f} m")
print(f"Rayleigh range = {zR_asym:.9f} m")
print(f"Waist radius = {w0_asym*1e3:.9f} mm")

Asymmetric cavity:
ROC1 = 1.000 m
ROC2 = 0.750 m

q eigenmode = -0.166666667+0.372677996j m
Waist position measured from M1 = 0.166666667 m
Rayleigh range = 0.372677996 m
Waist radius = 0.355273450 mm


## 10. What changed in the asymmetric cavity?

For the symmetric cavity,

\[
ROC_1=ROC_2
\]

and the waist lies at the geometric centre.

For the asymmetric cavity,

\[
ROC_1\neq ROC_2,
\]

and the ABCD eigenmode determines a new waist location and Rayleigh range.

Thus the important lesson is:

\[
\boxed{
\text{cavity geometry}
\rightarrow
M_{\rm RT}
\rightarrow
q_{\rm eigenmode}
\rightarrow
\text{waist position and size}
}.
\]

We obtained this without assuming that the waist lies at the centre.

This is the key reason the ABCD method is useful for the later four-mirror bow-tie cavity.

In [10]:
z_asym_array = np.linspace(0, L, 1000)

q_asym_array = np.array([
    propagate_q(
        q_asym,
        np.array([[1.0, distance],
                  [0.0, 1.0]])
    )
    for distance in z_asym_array
])

w_asym_array = np.array([
    beam_radius(q, lambda0)
    for q in q_asym_array
])

plt.figure(figsize=(9, 4))
plt.plot(z_asym_array, w_asym_array*1e3)
plt.axvline(
    (-z_asym)*1e0,
    linestyle="--",
    alpha=0.6,
    label="Waist"
)
plt.xlabel("Position from M1 [m]")
plt.ylabel("Beam radius w(z) [mm]")
plt.title("Gaussian eigenmode of the asymmetric Fabry–Pérot cavity")
plt.legend()
plt.grid(True)
plt.show()

/tmp/ipykernel_1055/1805737993.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Connection to the later bow-tie cavity

The four-mirror bow-tie cavity cannot in general be reduced to the simple symmetric two-mirror formulas.

The ABCD method gives us the required general workflow:

\[
\boxed{
\text{define each optical element}
\rightarrow
\text{multiply matrices}
\rightarrow
M_{\rm RT}
\rightarrow
\text{solve for }q
\rightarrow
\text{propagate }q
\rightarrow
w(s)
}.
\]

For oblique incidence on curved mirrors, the tangential and sagittal planes can acquire different effective curvatures. This will be treated explicitly in the bow-tie cavity model.

The passive bow-tie cavity notebook therefore builds directly on the method established here.